# Writing client code: notebooks and apps

This notebook is for whoever writes code against `ess.dispatch`: a notebook, a local app, a service-backed app, or a program.
It follows one thread, from a for loop over runs to a sum of runs that grows as they arrive.
Every step shows what the framework guarantees, and what the client must do in return.

The workflows are the toy SANS reduction of `toy_sans.py`.
A run's file holds counts per pixel and a monitor count.
`IOFQ` reduces one sample run and one can run to an I(Q) curve, `IOFQ_MULTI` sums several of each, and `FORWARD` reads a curve and returns its first point.
`FIND_CENTRE` finds the beam centre from a sample run, and `TRANSMISSION` computes a sample's transmission from its transmission run.
[workflow-authors.ipynb](workflow-authors.ipynb) shows how these specs and their bindings are written.
Here they are given, as an installed package would give them.

A spec has *params*, everything the workflow takes, and *outputs*.
A request gives each value in one of two forms: *inline* or *by reference*.
An *inline value* is the value itself in the request, such as `Reduction(bins=4)`; the record holds its JSON form.
A *reference* names data that exist elsewhere: a dataset, an output of a record, or an output of an accumulator.
The type at each *place*, the value of a parameter, an element of a list, or a field of a group, at any depth, decides which of the two it takes: a file takes a reference to a dataset only; a *group*, a pydantic model such as `Reduction` or `BeamCentre`, takes an inline value or a reference to an output; any other type, such as an `int`, takes an inline value only.
A group with a field that is a scipp variable, such as `IofQ`, has no inline form, so a request gives it by reference only.
A group, such as `Sample`, a sample run and its transmission, is given as an object of its fields, each inline or by reference, at any depth.

| Spec | Params | Outputs |
|---|---|---|
| `IOFQ` | `reduction: Reduction` (`bins=2`), `beam_centre: BeamCentre` (`x=0, y=0`), `sample: Sample`, `can: Can` | `iofq: IofQ` |
| `IOFQ_MULTI` | `reduction: Reduction`, `beam_centre: BeamCentre`, `sample: list[Sample]`, `can: list[Can]` | `iofq: IofQ` |
| `FORWARD` | `curve: IofQ` | `i0: sc.Variable` |
| `FIND_CENTRE` | `sample: Run` | `centre: BeamCentre` |
| `TRANSMISSION` | `run: Run` | `transmission: TransmissionFraction` |

Every parameter is a group, and a file is a field of one: `Run` is a group of a run's `file`, and `Sample` and `Can` are runs too.
`IofQ` is a group whose field `intensity` is the curve, a scipp variable, and `i0` is a scipp variable.
`Sample` has the fields `file` and `transmission: TransmissionFraction`, with no default: a request gives a transmission, typed in, `{'value': 0.9}`, or computed by `TRANSMISSION`, and its record says which.
`IOFQ_MULTI` has the settings of `IOFQ`, a table of sample runs, each with its transmission, and a table of can runs (section 6).

In [ ]:
import tempfile
from pathlib import Path

from ess.dispatch import (
    Backend,
    Client,
    NotReadable,
    Request,
    SubmitError,
    Template,
    dataset,
)
from ess.dispatch.testing import FakeDatasets, FakeStore
from toy_sans import (
    BINDINGS,
    CAN_RUNS,
    FIND_CENTRE,
    FORWARD,
    IOFQ,
    IOFQ_MULTI,
    SAMPLE_RUNS,
    TRANSMISSION,
    TRANSMISSION_RUNS,
    BeamCentre,
    Reduction,
    TransmissionFraction,
)

## Setup

The framework reads raw data through a *dataset source*.
Here a fake one writes a few runs of proposal `p1` as files in a temporary folder, and a fake store stands in for disk.

In [ ]:
folder = tempfile.TemporaryDirectory()
datasets = FakeDatasets(proposal='p1', directory=Path(folder.name))
for run, data in SAMPLE_RUNS.items():
    datasets.measure(run, data, role='sample', temperature=f'{run + 149}K')
for run, data in CAN_RUNS.items():
    datasets.measure(run, data, role='can')
for run, data in TRANSMISSION_RUNS.items():
    datasets.measure(run, data, role='transmission')

backend = Backend(datasets, BINDINGS, store=FakeStore())
client = Client(backend, proposal='p1', submitter='alice')

`local(proposal='p1', datasets=datasets, bind=BINDINGS, store=...)` does the same in one call.
This notebook keeps `backend` to open a second client later, as another process would with `connect(url, proposal='p1')` on the service.
The service is not implemented; everything below runs in this process.

## 1. From a for loop to records

A notebook without the framework calls the reduction directly:

```python
curves = {}
for run in (101, 102, 103):
    curves[run] = reduce_iofq(sample=run, can=110, bins=2)
```

With the framework, each call becomes a request: a spec and its params, by name.
`dataset(run=101)` names a dataset; every parameter but the runs has a default, so the request leaves them out.
A `Sample` is given by its fields: `{'file': dataset(run=101), 'transmission': {'value': 0.8}}` gives its file and a transmission typed in.
A `Can` is given by its file.
`submit` returns a *record* at once, and the requests run in parallel.

In [ ]:
can = {'file': dataset(run=110)}
records = {}
for run in (101, 102, 103):
    sample = {'file': dataset(run=run), 'transmission': {'value': 0.8}}
    records[run] = client.submit(IOFQ, params={'sample': sample, 'can': can})
client.wait(records)
client.output(records[101], 'iofq')

A record holds the request as the framework accepted it: every parameter filled in, defaults included, and each dataset resolved from the name the user gave (`run:101`) to its identity.
It holds the group `sample` as an object of its fields.
It holds no output values.
A record never changes; only its status moves from `pending` to `completed`, `failed`, or `cancelled`.

In [ ]:
record = records[101]
record.request.params, client.status(record)

Every call takes one record, a list, or a dict, and returns the same shape:

In [ ]:
client.status(records)

**A request that cannot run is refused at submission**, with an error that names the field.
Each of its `errors` names the request by its key, `entry`, the field by its place, `loc`, and says why, `msg`.
If one request of a call is refused, none is submitted, so a batch with one typo starts nothing.

In [ ]:
sample = {'file': dataset(run=102), 'transmission': {'value': 0.8}}
batch = {
    'good': Request(IOFQ, params={'sample': sample, 'can': can}),
    'typo': Request(
        IOFQ, params={'reduction': {'bins': 0}, 'sample': sample, 'can': can}
    ),
}
try:
    client.submit(batch)
except SubmitError as error:
    print(error)
    for e in error.errors:
        print(e.entry, e.loc, e.msg)

A group may be given as a dict of its fields, as above, or as an instance of its model, which a type checker checks (section 2); both make the same request.
A request whose values are all given inline may also give its params as an instance of the params model.
`Request(spec, params=...)` makes a request without submitting it, so that several go into one call.

A file is given by reference to a dataset only: by run, by PID, or by `dataset(path=...)`, so that the dataset source checks access and every file a request reads has an identity in the record; a path given as a string is refused:

In [ ]:
try:
    client.submit(
        IOFQ,
        params={
            'sample': {'file': '/data/run-102.json', 'transmission': {'value': 0.8}},
            'can': can,
        },
    )
except SubmitError as error:
    print(error)

## 2. Chains: a reference to a pending record

A value may also be an output of a record, given as `record.ref(output)`.
The record may still be pending, so a chain is submitted without waiting.
The framework runs `FORWARD` once the I(Q) has completed.

In [ ]:
curve = client.submit(
    IOFQ,
    params={
        'reduction': Reduction(bins=4),
        'sample': {'file': dataset(run=103), 'transmission': {'value': 0.8}},
        'can': can,
    },
)
i0 = client.submit(FORWARD, params={'curve': curve.ref('iofq')})
client.output(i0, 'i0')

`client.output` waits for the record.
The framework checks at submission that the output *fits* the place, by their types: an `IofQ` output fits a place typed `IofQ`.
A reference stands for a group or a file only, since a group names a role: a place typed `sc.Variable` would take every output of that type, so a picker of records would list unrelated outputs.
The output `i0` of `FORWARD`, a scipp variable, does not fit `IofQ`:

In [ ]:
try:
    client.submit(FORWARD, params={'curve': i0.ref('i0')})
except SubmitError as error:
    print(error)

A group such as `BeamCentre` takes an inline value or a reference, whichever the request gives.
One user types in the beam centre; another computes it with `FIND_CENTRE`, whose output `centre` is a `BeamCentre`:

In [ ]:
finder = client.submit(FIND_CENTRE, params={'sample': {'file': dataset(run=103)}})
sample = {'file': dataset(run=103), 'transmission': {'value': 0.8}}
runs = {'sample': sample, 'can': can}
centred = {
    'inline': client.submit(
        IOFQ, params={'beam_centre': BeamCentre(x=1.25, y=0.0), **runs}
    ),
    'reference': client.submit(
        IOFQ, params={'beam_centre': finder.ref('centre'), **runs}
    ),
}
print(client.output(finder, 'centre'))
{form: client.output(r, 'iofq').intensity.values for form, r in centred.items()}

The workflow gets a `BeamCentre` either way, and the finder found (1.25, 0), so both give the same I(Q).
Each record holds the value in the form the request gave it, inline or as a reference, so the record of the second says where its beam centre came from:

In [ ]:
[centred[form].request.params['beam_centre'] for form in centred]

## 3. Who keeps a value

An output value, such as the I(Q) array, lives while something keeps it, and nothing else keeps it:

| Keeper | Keeps the value until |
|---|---|
| the client that submitted the record | the client releases the record, or ends |
| a pending request that reads it | that request's workflow has returned |
| a stage, an accumulator's template, or a push that references it | the binding has taken it, or the stage, accumulator, or push is released or stopped |
| a persist request | the value is written |
| the store, once the value is written | the proposal's history is dropped |

A record object, a reference, or a label keeps nothing.
Releasing drops the client's hold; the record stays, but its value is gone for good:

In [ ]:
client.release(curve)
try:
    client.output(curve, 'iofq')
except NotReadable as error:
    print(error)

`FORWARD` had read it already, so `i0` is unaffected.
Had `FORWARD` still been pending, it would have kept the curve until it ran.

Two consequences that a client author must know:

- **Work that nothing keeps is cancelled.** Releasing a pending record that no pending request reads cancels it, and closing a client releases everything it keeps. A notebook that submits a batch and closes its client before the batch has run cancels the batch, unless it was persisted.
- **Another client never reads the values this client keeps**, not even in the same process. It reads only persisted values.

On the service, the memory a client keeps is capped, and a client that makes no call for a while is ended (its *lease*).
A long-lived app therefore releases what it no longer shows.

## 4. Persist, and find it the next day

`persist=` at submission makes the store the only keeper.
This is how an unattended batch is submitted: the client keeps nothing, and the records complete once their outputs are written.
A *label* names the batch, and a dict's keys become the *members*:

In [ ]:
night = client.submit(
    {
        f'{run + 149}K': Request(
            IOFQ,
            params={
                'sample': {'file': dataset(run=run), 'transmission': {'value': 0.8}},
                'can': can,
            },
        )
        for run in (101, 102, 103)
    },
    label='scan',
    persist=True,
)
client.wait(night)

The next morning, another client of the same proposal finds the batch by its label and reads from the store.
Here a second client in this process stands in for it:

In [ ]:
morning = Client(backend, proposal='p1', submitter='bob')
latest = morning.latest('scan', member='250K')
morning.output(latest, 'iofq')

`client.persist(record)` instead adds the store next to the client, for a record the client keeps.
On the service, records are the proposal's history: they last until the proposal has been idle for days to weeks, and are then dropped as a whole.
With `local()`, history is held in memory and ends with the process.
What must last longer is published to SciCat (not implemented).

## 5. Templates: tune with a stage

A *template* is a spec with some parameters fixed, and *blanks*, the names of the parameters left open.
It takes the fixed values as a request does, as `params=`.
A *stage* is a template the backend keeps for this client, so that what depends only on the fixed values is computed once.
Tuning `bins` then does not load the runs again:

In [ ]:
tune = client.stage(
    Template(
        IOFQ,
        params={
            'sample': {'file': dataset(run=101), 'transmission': {'value': 0.8}},
            'can': can,
        },
        blanks=('reduction',),
    )
)
for bins in (1, 2, 4):
    tuned = client.compute(tune, params={'reduction': Reduction(bins=bins)})
    print(bins, client.output(tuned, 'iofq').intensity.values)

A call through a stage makes the record of the plain request, the one with every value given.
Nothing in the record says it went through a stage, and its outputs are those of the plain request (the *caching* symmetry):

In [ ]:
tuned.request

A blank is a whole parameter, and a group is one key of the workflow's pipeline.
So a stage that tunes the transmission of run 101 has `sample` as its blank, and each call loads the run again.

A stage needs a binding that makes stages, such as one built from a sciline pipeline, which computes the fixed part once.
A stage over any other binding is refused when it is made:

In [ ]:
try:
    client.stage(Template(FORWARD, blanks=('curve',)))
except SubmitError as error:
    print(error)

## 6. Sum runs as they arrive: an accumulator

A *table* is a list parameter whose *rows* an accumulator takes a push at a time.
`IOFQ_MULTI` has two tables: `sample`, whose rows are `Sample` groups, and `can`, whose rows are can runs ([README, Combining runs](../README.md#combining-runs-tables)).

An *accumulator* is a template whose blanks are tables, kept by the backend for this client.

Each table is either a blank or fixed.
A fixed table is given whole in `params`, as a request gives it, and stays as given for the accumulator's life.
With one can for all samples, the template fixes `can` and leaves `sample` blank:

```python
Template(IOFQ_MULTI, params={'can': [can]}, blanks=('sample',))
```

A push into a fixed table is refused, and `client.accumulator` refuses a blank that is not a table.

Each *push* gives rows of one or more tables, by the name of each, as a list.
State n, the accumulator after n pushes, has the outputs of the plain request whose tables hold the rows of the first n pushes.

In [ ]:
total = client.accumulator(Template(IOFQ_MULTI, blanks=('sample', 'can')))
total.push(can=[can])  # state 1
total.push(
    sample=[{'file': dataset(run=101), 'transmission': {'value': 0.8}}]
)  # state 2
state_2 = client.output(total, 'iofq').intensity
measured = client.submit(TRANSMISSION, params={'run': {'file': dataset(run=120)}})
total.push(
    sample=[{'file': dataset(run=102), 'transmission': measured.ref('transmission')}]
)  # state 3
state_3 = client.output(total, 'iofq').intensity
state_2.values, state_3.values

`push` returns once the backend has logged it; the backend adds it later.
A read *pins* the state after the pushes logged so far and waits for it.
State 1 has no sample row, so a read of it would be refused: the table `sample` needs a row.

A value that differs per run is in the run's file, and a value that the file lacks or has wrong is a field of the run's row.
Each row has its own transmission: that of run 101 is typed in, and that of run 102 is computed from transmission run 120.
Each field of a row is given inline or by reference on its own, so the transmission of one run may be given inline, `TransmissionFraction(value=0.9)`, in the same push as a reference for another.

State 3 equals the plain request over the same rows (the *arrival* symmetry):

In [ ]:
plain = client.compute(
    IOFQ_MULTI,
    params={
        'sample': [
            {'file': dataset(run=101), 'transmission': TransmissionFraction(value=0.8)},
            {'file': dataset(run=102), 'transmission': TransmissionFraction(value=0.9)},
        ],
        'can': [can],
    },
)
client.output(plain, 'iofq').intensity.values

With one row in each table, a request of `IOFQ_MULTI` gives the outputs of the request of `IOFQ` with the same values (the *one-row* symmetry).

Behind the accumulator, the binding of `IOFQ_MULTI` makes a *driver*, which the backend keeps for the accumulator.
The driver holds the sums of numerators and denominators, and a push loads its runs and adds them.
Only a spec whose binding makes drivers has accumulators; the backend refuses any other when the accumulator is made.

A push may give rows of several tables, which enter one state, as the push of runs 103 and 111 below does.
The tables stand side by side, as in a request: rows of different tables have nothing to do with each other, and the first sample row is not paired with the first can row.
Values that belong to one run, such as its transmission, are fields of its row.
A push may give several rows of its table, which enter one state; a push of no rows is refused.

**Readers never see a value change.**
`client.output` of an accumulator returns a copy, so `state_2` above still holds state 2.
A request may read a state too, through `total.ref(...)`.
It reads the output of the driver without a copy, so the backend holds back the next push until that request's workflow has returned:

In [ ]:
forward_3 = client.submit(FORWARD, params={'curve': total.ref('iofq')})  # pins state 3
total.push(
    sample=[{'file': dataset(run=103), 'transmission': {'value': 0.6}}],
    can=[{'file': dataset(run=111)}],
)  # state 4
client.output(forward_3, 'i0').value, state_3['Q', 0].value

The order of the calls is the order of the states, as if each call ran after the one before it, although pushes and reads run in parallel.

The rows of a table are distinct.
A row pushed again is refused, since pushing a run twice is usually a cell run twice.
Rows are compared whole, as the record holds them, so run 103 with another transmission would be another row.
For a table whose rows have a JSON form, such as time slices, the backend also compares the rows read: a slice given inline and a reference to an equal output are one row, and the repeat fails the push, which stops the accumulator.

In [ ]:
try:
    total.push(sample=[{'file': dataset(run=103), 'transmission': {'value': 0.6}}])
except SubmitError as error:
    print(error)

At the end, `freeze` turns the last state into an ordinary record, without a copy, and the accumulator takes no more pushes:

In [ ]:
summed = client.freeze(total)
client.output(summed, 'iofq').intensity.values

## 7. Where a result came from

Every record can say which requests and datasets produced it.
The record of a frozen state holds the plain request over its rows, each `Sample` as an object of its fields, and does not name the accumulator.
The transmission of run 102 is a reference to the record of `TRANSMISSION`:

In [ ]:
provenance = client.provenance(summed)
print(summed.request.params['sample'][1])
[str(d) for d in provenance.datasets()]

## 8. Flavours of client code

Everything above is the same API for every client.
What differs between flavours is which calls they rely on, and what they must take care of.

| Flavour | Makes its client with | Typically | Must take care of |
|---|---|---|---|
| notebook | `local(...)` | `submit`, `output`, stages, accumulators | releasing large values; not modifying values that `output` returned (see below) |
| local app (not a notebook) | `local(...)` | the same, behind a form | releasing what it no longer shows; choosing the rows of each table |
| app against the service | `connect(url, proposal=...)` (not implemented) | `submit(..., persist=True)`, labels, `latest` | the memory cap and the lease; reading results from the store |
| batch or automatic reduction | either | `apply`, `Rule`, `TriggerLoop` | choosing what to persist |
| program, CLI, agent | `connect(...)` (not implemented) | `SpecId`, `SerializedWorkflowSpec` | importing no workflow package |

Points that hold for apps but not for notebooks:

- **Forms come from the spec.** `IOFQ.serialize()` gives the schema of the params: the JSON Schema of each parameter that takes an inline value, for a form, and the type of data that fits each reference, such as `Path` for a file or `BeamCentre` for a group, for a picker of datasets or records. An app that imports only a package's specs, kept in a module apart from the workflow code, builds forms without the workflow's dependencies ([ADR 0007](../adr/0007-packages-split-by-dependencies.md)).
- **Errors name the field at fault.** Each of the `errors` of a `SubmitError` is a `FieldError`: `entry`, the key or index of the request in the call (`'typo'` above), `loc`, the place of the field (`('reduction', 'bins')`), and `msg`. A form shows each next to its field. A read raises `NotReadable` for a record or an accumulator that exists but cannot be read, such as one the client released, `KeyError` for one it does not know, and `RecordFailed` for a record that failed or was cancelled.
- **An app that does not import the workflow package** names the spec by `SpecId(name='toy-sans-iofq', version=1)` and gets the same checks.
- **The framework knows no technique.** Which sample run goes with which can run, and which runs form the rows of a table, is the app's decision.

**Batch reduction** fills a template once per dataset with `apply`: each dataset fills the place `at` in the one blank, here the file of the sample run whose beam centre `FIND_CENTRE` finds.
`fields` gives the blank's other fields, the same for each dataset, such as `fields={'thickness': 0.2}` for a sample group with a thickness, shared by a batch of samples in one kind of cell; a field left out takes its default.
A `Run` has no other field.
`apply` submits nothing and returns the requests keyed by a metadata field:

In [ ]:
from ess.dispatch import Selector, apply

samples = client.datasets.list(Selector(role='sample'))
requests = apply(
    Template(FIND_CENTRE, blanks=('sample',)),
    samples,
    client.datasets,
    at=('sample', 'file'),
    member_field='temperature',
)
batch = client.submit(requests, label='batch', persist=True)
client.wait(batch)
sorted(r.member for r in client.records(label='batch'))

A `Rule` is plain data: a name, a template, a selector, a label, the place `at`, and optionally the blank's other `fields`, a `member_field`, a `lookup`, a `series`, and `persist`, the outputs it persists. The `TriggerLoop` fills its template for each new dataset that the selector matches, and submits it; see [automatic-reduction.md](../automatic-reduction.md).

**Values in the user's process are not copied.**
In a notebook, `client.output` of a record returns the value the framework holds, not a copy.
Modifying it in place changes what later requests read and what `persist` writes.
This is open question 7 of the [README](../README.md); until it is settled, treat outputs as read-only.

## Summary: what client code relies on, and what it owes

| Relies on | Owes |
|---|---|
| a record never changes, only its status | keeping (or persisting) what it still needs, and releasing the rest |
| a refused call submits nothing, and names the field | treating output values as read-only |
| a value lives while something keeps it, and only then | giving each file by reference, through the dataset source |
| a request runs once the records it references have completed | choosing the rows of each table: the framework knows no technique |
| a value is given inline or by reference, as its type allows; a reference is checked by type at submission, and the workflow gets the data it names | |
| a reader never sees a value change | |
| a call through a stage, or a state of an accumulator, gives the outputs of its plain request | |
| where a request runs changes neither its record nor its outputs | |

[framework-promises.ipynb](framework-promises.ipynb) shows what keeping these promises takes, in this process and on the service.

In [ ]:
client.close()
morning.close()
backend.close()
folder.cleanup()